In [2]:
# Import required libraries
import sys
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime
from tqdm import tqdm

# Import the LLM debiasing analyzer
from LLM_debias import LLMPositionBiasAnalyzer

print("💄 Libraries imported successfully!")
print(f"📅 Experiment started at: {datetime.now()}")


💄 Libraries imported successfully!
📅 Experiment started at: 2025-08-10 09:48:28.728872


In [24]:
import pandas as pd
import json
from tqdm import tqdm

meta_path = "data/beauty/meta_All_Beauty.jsonl"
reviews_path = "data/beauty/All_Beauty.jsonl"
output_path = "data/beauty/user_title_timestamp.csv"

# ===== 1. Read metadata file and store title + details =====
meta_records = []

with open(meta_path, "r") as f:
    for line in tqdm(f, desc="Reading metadata"):
        line = line.strip()
        if not line:
            continue  # skip empty lines
        try:
            data = json.loads(line)
        except json.JSONDecodeError:
            continue  # skip malformed JSON lines
        
        parent_asin = data.get("parent_asin")
        title = data.get("title", "").strip()
        details = data.get("details", {})
        
        # Convert 'details' dict to string
        details_str = ""
        if isinstance(details, dict):
            details_str = " | ".join(f"{k}: {v}" for k, v in details.items())
        elif details:
            details_str = str(details)

        if parent_asin and title:
            meta_records.append({
                "parent_asin": parent_asin,
                "Title": title,
                "Details": details_str
            })

meta_df = pd.DataFrame(meta_records)

# ===== 2. Read reviews file =====
reviews = []
with open(reviews_path, "r") as f:
    for line in tqdm(f, desc="Reading reviews"):
        line = line.strip()
        if not line:
            continue
        try:
            data = json.loads(line)
        except json.JSONDecodeError:
            continue
        reviews.append(data)

reviews_df = pd.DataFrame(reviews)

# ===== 3. Merge on parent_asin =====
df = reviews_df.merge(meta_df, on="parent_asin", how="left")

# ===== 4. Append details to title =====
df["Title"] = df["Title"] + df["Details"].apply(lambda d: f" | {d}" if pd.notna(d) and d else "")

# ===== 5. Drop rows with no title =====
filtered_df = df.dropna(subset=["Title"])

# ===== 6. Rename columns =====
filtered_df = filtered_df.rename(columns={
    "user_id": "UserID",
    "timestamp": "Timestamp"
})

# ===== 7. Select final columns =====
final_df = filtered_df[["UserID", "Title", "Timestamp"]]

# ===== 8. Save to CSV =====
final_df.to_csv(output_path, index=False)
print(f"💾 Saved matched user-product-timestamp data to {output_path}")
print(final_df.head())


Reading metadata: 112592it [00:01, 104270.96it/s]
Reading reviews: 701528it [00:04, 151503.63it/s]


💾 Saved matched user-product-timestamp data to data/beauty/user_title_timestamp.csv
                         UserID  \
0  AGKHLEW2SOWHNMFQIJGBECAF7INQ   
1  AGKHLEW2SOWHNMFQIJGBECAF7INQ   
2  AE74DYR3QUGVPZJ3P7RFWBGIX7XQ   
3  AFQLNQNQYFWQZPJQZS6V3NZU4QBQ   
4  AFQLNQNQYFWQZPJQZS6V3NZU4QBQ   

                                               Title      Timestamp  
0  Herbivore - Natural Sea Mist Texturizing Salt ...  1588687728923  
1  All Natural Vegan Dry Shampoo Powder - Eco Fri...  1588615855070  
2  New Road Beauty - Creamsicle - Variety 3 Pack ...  1589665266052  
3  muaowig Ombre Body Wave Bundles 1B Grey Human ...  1643393630220  
4  Yinhua Electric Nail Drill Kit Portable Profes...  1609322563534  


In [4]:
# Load processed data (run this if data preprocessing was done previously)
final_df = pd.read_csv('data/beauty/user_title_timestamp.csv')
print(f"📊 Loaded processed beauty data: {len(final_df):,} records")
print(f"📈 Unique users: {final_df['UserID'].nunique():,}")
print(f"💄 Unique products: {final_df['Title'].nunique():,}")
print("\n📊 Sample of processed data:")
print(final_df.head())


📊 Loaded processed beauty data: 701,444 records
📈 Unique users: 631,915
💄 Unique products: 112,013

📊 Sample of processed data:
                         UserID  \
0  AGKHLEW2SOWHNMFQIJGBECAF7INQ   
1  AGKHLEW2SOWHNMFQIJGBECAF7INQ   
2  AE74DYR3QUGVPZJ3P7RFWBGIX7XQ   
3  AFQLNQNQYFWQZPJQZS6V3NZU4QBQ   
4  AFQLNQNQYFWQZPJQZS6V3NZU4QBQ   

                                               Title      Timestamp  
0  Herbivore - Natural Sea Mist Texturizing Salt ...  1588687728923  
1  All Natural Vegan Dry Shampoo Powder - Eco Fri...  1588615855070  
2  New Road Beauty - Creamsicle - Variety 3 Pack ...  1589665266052  
3  muaowig Ombre Body Wave Bundles 1B Grey Human ...  1643393630220  
4  Yinhua Electric Nail Drill Kit Portable Profes...  1609322563534  


In [5]:
# Initialize the LLM Position Bias Analyzer for beauty products
print("🚀 Initializing LLM Position Bias Analyzer for Beauty Products...")

analyzer = LLMPositionBiasAnalyzer(
    data=final_df,
    data_name="beauty",
    model="gpt-3.5-turbo",
    backend="openai",
    list_size=20,
    api_tier="tier_2"
)

print("✅ Analyzer initialized successfully!")


🚀 Initializing LLM Position Bias Analyzer for Beauty Products...
📊 User filtering results:
  Total users in dataset: 631915
  Users with ≥6 items: 1061
  Filtered out: 630854 users
✅ Selected 5 bias users and 200 evaluation users
   All selected users have ≥6 items for reliable evaluation
Initialized LLM Bias Analyzer:
  Model: gpt-3.5-turbo
  Backend: openai
  API Tier: tier_2
  Rate Limits: 5000 RPM, 2000000 TPM
  Max Workers: 25
  Batch Size: 50
  Request Delay: 0.030s
✅ Analyzer initialized successfully!


In [27]:
# # Compute bias analysis with 5 bias users (list size 20)
# print("🔍 Computing bias analysis with 5 bias users...")

# # Uncomment the line below to run bias analysis
bias_analysis = analyzer.compute_bias_analysis(5, None, True, None, 20)
print(bias_analysis)


Bias users: ['AGTQOBCMPQJSDPCPVQSRF7IMJLFA', 'AGIE52RDMRZU6FJGVXIX3MUUC3AA', 'AHCZPHIPKSBSHMF7WLBPVQSEWM5A', 'AH443J7EG4DKGMUFINDIN2F75VVQ', 'AF4T3AQXLGSWBXPJV3RI2DBPEYUA']
\nCalculating bias scores...


Bias detection:   0%|                                     | 0/5 [00:00<?, ?it/s]


Running bias detection experiment with 50 shuffles...
Executing 50 shuffles in parallel with max_workers=25...
Rate limiting: 25 workers, 0.030s delay, batch size: 50



Bias detection:  20%|█████▊                       | 1/5 [00:05<00:21,  5.31s/it]

Completed 50 successful shuffles out of 50 attempted
Bias Detection Results:
  Average primacy items in top 10%: 0.500
  Average recency items in top 10%: 0.740
  Average middle items in top 10%: 0.760

Running bias detection experiment with 50 shuffles...
Executing 50 shuffles in parallel with max_workers=25...
Rate limiting: 25 workers, 0.030s delay, batch size: 50


...

Completed 50 successful shuffles out of 50 attempted
Bias Detection Results:
  Average primacy items in top 10%: 0.620
  Average recency items in top 10%: 0.500
  Average middle items in top 10%: 0.880

Running bias detection experiment with 50 shuffles...
Executing 50 shuffles in parallel with max_workers=25...
Rate limiting: 25 workers, 0.030s delay, batch size: 50



Bias detection:  80%|███████████████████████▏     | 4/5 [00:21<00:05,  5.38s/it]

Completed 50 successful shuffles out of 50 attempted
Bias Detection Results:
  Average primacy items in top 10%: 0.500
  Average recency items in top 10%: 0.560
  Average middle items in top 10%: 0.940

Running bias detection experiment with 50 shuffles...
Executing 50 shuffles in parallel with max_workers=25...
Rate limiting: 25 workers, 0.030s delay, batch size: 50



Bias detection: 100%|█████████████████████████████| 5/5 [00:26<00:00,  5.35s/it]

Completed 50 successful shuffles out of 50 attempted
Bias Detection Results:
  Average primacy items in top 10%: 0.300
  Average recency items in top 10%: 0.520
  Average middle items in top 10%: 1.180
{'bias_scores': {'avg_primacy': np.float64(0.5279999999999999), 'avg_recency': np.float64(0.6), 'avg_middle': np.float64(0.8719999999999999)}, 'propensity_scores': {1: 0.9455391358903965, 2: 0.9626649397806111, 3: 0.977324749017514, 4: 0.9893973290293935, 5: 0.9987819059145051, 6: 1.0053995669490867, 7: 1.0091943654360311, 8: 1.0101341082589959, 9: 1.0082108106358734, 10: 1.0034408090868516, 11: 0.9958645303816003, 12: 0.9855459210362711, 13: 0.9725715486209945, 14: 0.9570493925430518, 15: 0.9391073479293851, 16: 0.9188914715977974, 17: 0.896564003750481, 18: 0.8723012028397289, 19: 0.8462910339619841, 20: 0.8187307530779818}, 'avg_bias_result': {'avg_primacy': np.float64(0.5279999999999999), 'avg_recency': np.float64(0.6), 'avg_middle': np.float64(0.8719999999999999)}, 'experiment_resul

In [6]:
prebias_gpt35_beauty = {
    'avg_primacy': 0.404,
    'avg_recency': 0.556,
    'avg_middle': 1.04
}

prebias_gpt4o_beauty = {
    'avg_primacy': 0.772,
    'avg_recency': 0.592,
    'avg_middle': 0.636
}

prebias_gpt35_beauty_2 = {
    'avg_primacy': 0.5279999999999999,
    'avg_recency': 0.6,
    'avg_middle': 0.871999999999999
}

In [7]:
# Main debiasing experiment with list size 20
print("\n🔧 COMPLETE DEBIASING EXPERIMENT - BEAUTY PRODUCTS")
print("=" * 55)

num_candidates = 20    # Number of candidates per evaluation
num_trials = 25        # Number of randomization trials per user
batch_size = 20        # Batch size for processing

print(f"🎯 Candidates per evaluation: {num_candidates}")
print(f"🔄 Trials per user: {num_trials}")
print(f"📦 Batch size: {batch_size}")

# Uncomment and run the evaluation below
results = analyzer.evaluate_our_method_batched(
    num_candidates=num_candidates,
    num_trials=num_trials,
    aggregation_method="mean",
    use_parallel=True,
    precalculated_bias=prebias_gpt35_beauty_2,
    checkpoint_file="evaluation_checkpoint_beauty_gpt35_4.json"
)


🔧 COMPLETE DEBIASING EXPERIMENT - BEAUTY PRODUCTS
🎯 Candidates per evaluation: 20
🔄 Trials per user: 25
📦 Batch size: 20
📁 Checkpoint file: evaluation_checkpoint_beauty_gpt35_4.json
API Tier: tier_2 (RPM: 5000, TPM: 2000000)
Max workers - Bias: 25, Trials: 12, Users: 3
🔄 Recalculating bias analysis with new precalculated bias scores...
   Previous bias: {}
   New bias: {'avg_primacy': 0.5279999999999999, 'avg_recency': 0.6, 'avg_middle': 0.871999999999999}
Bias users: ['AEAXVFSE6HQTWP5RA65SLOATKU4Q', 'AF7UXRQ6VLFJ3MFLZQYVNUNBI7OQ', 'AF7N64YSCYUJ4PMFFF4KXI4AD4FA', 'AGP4FD5ADGRFJAE6PQ2ZAUVJVHWA', 'AFOKWZIJI7K6Z6R4T7SSXRTWZNOQ']
Using precalculated bias scores...
👥 Total users: 200, Completed: 0, Remaining: 200

🔄 Processing batch 1/8 (25 users)
Evaluating 25 users in parallel with max_workers=3...

Running 25 randomization trials (with raw data preservation)...
Executing 25 trials in parallel with max_workers=12...
Rate limiting: 12 workers, 0.030s delay, batch size: 50

Running 25 rand

Trials (batch 1/1):   0%|                                | 0/25 [00:00<?, ?it/s]

Trials (batch 1/1):   0%|                                | 0/25 [00:00<?, ?it/s]

Trials (batch 1/1):   8%|█▉                      | 2/25 [00:01<00:15,  1.44it/s]

Trials (batch 1/1):  16%|███▊                    | 4/25 [00:01<00:06,  3.26it/s]

Trials (batch 1/1):  24%|█████▊                  | 6/25 [00:01<00:03,  4.97it/s]

Trials (batch 1/1):  44%|██████████             | 11/25 [00:02<00:01, 10.64it/s]

Trials (batch 1/1):  64%|██████████████▋        | 16/25 [00:03<00:01,  5.85it/s]

Trials (batch 1/1):  68%|███████████████▋       | 17/25 [00:03<00:01,  5.71it/s]

Trials (batch 1/1):  84%|███████████████████▎   | 21/25 [00:03<00:00,  7.66it/s]

Trials (batch 1/1):  88%|████████████████████▏  | 22/25 [00:04<00:00,  7.79it/s]

Trials (batch 1/1): 100%|███████████████████████| 25/25 [00:04<00:00,  5.08it/s]


Completed 25 successful trials out of 25 attempted

Running 25 randomization trials (with raw data preservation)...
Executing 25 trials in parallel with max_workers=12...
Rate limiting: 12 workers, 0.030s delay, batch size: 50



Trials (batch 1/1): 100%|███████████████████████| 25/25 [00:05<00:00,  4.45it/s]


Completed 25 successful trials out of 25 attempted


...

Completed 25 successful trials out of 25 attempted



Trials (batch 1/1):  64%|██████████████▋        | 16/25 [00:02<00:01,  7.54it/s]

Trials (batch 1/1):  72%|████████████████▌      | 18/25 [00:02<00:00,  8.57it/s]

Trials (batch 1/1):  20%|████▊                   | 5/25 [00:01<00:03,  5.82it/s]

Trials (batch 1/1):  80%|██████████████████▍    | 20/25 [00:03<00:00,  7.07it/s]

Trials (batch 1/1):  36%|████████▋               | 9/25 [00:01<00:01,  8.40it/s]

Trials (batch 1/1): 100%|███████████████████████| 25/25 [00:03<00:00,  6.27it/s]


Trials (batch 1/1):  56%|████████████▉          | 14/25 [00:02<00:02,  4.89it/s]

Completed 25 successful trials out of 25 attempted




Trials (batch 1/1):  64%|██████████████▋        | 16/25 [00:02<00:01,  5.92it/s]

Trials (batch 1/1):  72%|████████████████▌      | 18/25 [00:02<00:00,  7.15it/s]

Trials (batch 1/1):  80%|██████████████████▍    | 20/25 [00:03<00:00,  7.40it/s]

Trials (batch 1/1):  92%|█████████████████████▏ | 23/25 [00:03<00:00, 10.48it/s]

Trials (batch 1/1): 100%|███████████████████████| 25/25 [00:03<00:00,  6.28it/s]


Completed 25 successful trials out of 25 attempted
Progress: 25/25 users completed
✅ Completed evaluation of 25 users
✅ Batch 8 completed. Progress: 200/200 users

📊 Computing final metrics from 200 user results...

OUR METHOD EVALUATION RESULTS vs BENCHMARKS

Our Method Results:
  Accuracy:    0.1100 ± 0.3129
  NDCG@1:      0.1100 ± 0.3129
  NDCG@5:      0.2734 ± 0.3418
  NDCG@10:     0.3428 ± 0.3108
  NDCG@20:     0.4280 ± 0.2295
  Number of evaluations: 200

Benchmark Results (Accuracy) - From Paper:
Method          Movie Dataset  
------------------------------
Raw Output      0.2740±0.0593
Bootstrapping   0.2537
STELLA          0.2976
Our Method      0.1100±0.3129

Accuracy Comparison (Movie Dataset):
----------------------------------------
Our Method vs Raw Output:    -0.1640
Our Method vs Bootstrapping: -0.1437
Our Method vs STELLA:        -0.1876

NDCG Analysis:
----------------------------------------
NDCG@1 = Accuracy: 0.1100
NDCG@5:  0.2734 (248.6% of NDCG@1)
NDCG@10: 0.342